In [ ]:
import math
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

N, CTX = 4, 3                      # 4-gram: 3 previous words -> next word

# ---------- Step 0: corpus ----------
text = """
the students opened their books
the students opened their exams
the students opened their books
the students opened their laptops
the proctor started the clock
the teacher opened the door
the students closed their books
the proctor started the exam
"""
sentences = [line.lower().split() for line in text.strip().split("\n")]

# ---------- Step 1: vocabulary ----------
word_counts = Counter(w for s in sentences for w in s)
vocab = ["<s>", "</s>", "<unk>"] + sorted(word_counts)
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
to_id = lambda w: w2i.get(w, w2i["<unk>"])
print("V =", V, vocab)

# ---------- Step 2: (3 previous words -> next word) tensors ----------
def encode(words):
    return [w2i["<s>"]] * CTX + [to_id(w) for w in words] + [w2i["</s>"]]

all_ids = [encode(s) for s in sentences]
X = torch.tensor([ids[t - CTX:t] for ids in all_ids for t in range(CTX, len(ids))])  # [P, 3]
Y = torch.tensor([ids[t]         for ids in all_ids for t in range(CTX, len(ids))])  # [P]
print("pairs:", len(Y), " X:", tuple(X.shape), " Y:", tuple(Y.shape))

def make_ctx(words):
    return ([w2i["<s>"]] * CTX + [to_id(w) for w in words])[-CTX:]

# ---------- Model A: count n-gram, pure tensors, with backoff ----------
class CountNGram:
    def __init__(self, X, Y, V):
        c4 = torch.zeros(V, V, V, V)                 # c4[a,b,c,w] = times (a,b,c) was followed by w
        c4.index_put_((X[:, 0], X[:, 1], X[:, 2], Y), torch.ones(len(Y)), accumulate=True)
        # summing out the OLDEST dimensions leaves counts for shorter contexts
        self.c = {3: c4,                             # [V,V,V,V] 3-word context
                  2: c4.sum(0),                      # [V,V,V]   2-word context
                  1: c4.sum((0, 1)),                 # [V,V]     1-word context
                  0: c4.sum((0, 1, 2))}              # [V]       no context (unigram)

    def probs(self, ctx):                            # ctx: list of 3 word ids
        for k in (3, 2, 1, 0):                       # backoff: long context first
            row = self.c[k][tuple(ctx[CTX - k:])]    # counts of every next word for this context
            if row.sum() > 0:
                return row / row.sum(), k            # count(context + w) / count(context)

count_lm = CountNGram(X, Y, V)

# ---------- Model B: neural window model ----------
class NgramNN(nn.Module):
    def __init__(self, V, ctx=3, emb_dim=16, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb_dim)
        self.hidden = nn.Linear(ctx * emb_dim, hidden)
        self.W = nn.Linear(hidden, V)

    def forward(self, ctx_ids):                      # [B, 3]
        e = self.emb(ctx_ids).flatten(1)             # [B, 3*emb_dim]  concatenate
        x = torch.tanh(self.hidden(e))               # prefix vector x
        return self.W(x)                             # logits [B, V]

loader = DataLoader(TensorDataset(X, Y), batch_size=16, shuffle=True)
model = NgramNN(V, CTX).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(301):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = F.cross_entropy(model(xb), yb)        # softmax + -log P(correct word)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(yb)
    if epoch % 50 == 0:
        avg = total / len(Y)
        print(f"epoch {epoch:3d}  loss {avg:.4f}  perplexity {math.exp(avg):.2f}")

@torch.no_grad()
def nn_probs(ctx):
    model.eval()
    logits = model(torch.tensor([ctx], device=device))
    return F.softmax(logits, dim=-1)[0].cpu()

# ---------- Step 3: predict and generate ----------
def show_top(name, p, k=4):
    vals, idx = torch.topk(p, k)
    print(f"   {name}:", [(vocab[i], round(v.item(), 3)) for v, i in zip(vals, idx)])

for phrase in ["the students opened their", "proctor opened their", "teacher closed their"]:
    ctx = make_ctx(phrase.split())
    p_c, used = count_lm.probs(ctx)
    print(f'\n"{phrase}"')
    show_top(f"count (used {used}-word context)", p_c)
    show_top("neural", nn_probs(ctx))

def generate(prob_fn, max_len=10):
    ctx, out = [w2i["<s>"]] * CTX, []
    for _ in range(max_len):
        p = prob_fn(ctx).clone()
        p[w2i["<s>"]] = 0; p[w2i["<unk>"]] = 0
        nxt = torch.multinomial(p, 1).item()
        if vocab[nxt] == "</s>": break
        out.append(vocab[nxt])
        ctx = ctx[1:] + [nxt]                        # slide the window
    return " ".join(out)

print("\ncount  :", generate(lambda c: count_lm.probs(c)[0]))
print("neural :", generate(nn_probs))

V = 17 ['<s>', '</s>', '<unk>', 'books', 'clock', 'closed', 'door', 'exam', 'exams', 'laptops', 'opened', 'proctor', 'started', 'students', 'teacher', 'the', 'their']
pairs: 48  X: (48, 3)  Y: (48,)
epoch   0  loss 2.5562  perplexity 12.89
epoch  50  loss 0.3460  perplexity 1.41
epoch 100  loss 0.3345  perplexity 1.40
epoch 150  loss 0.3268  perplexity 1.39
epoch 200  loss 0.3438  perplexity 1.41
epoch 250  loss 0.3312  perplexity 1.39
epoch 300  loss 0.3242  perplexity 1.38

"the students opened their"
   count (used 3-word context): [('books', 0.5), ('laptops', 0.25), ('exams', 0.25), ('proctor', 0.0)]
   neural: [('books', 0.477), ('laptops', 0.291), ('exams', 0.232), ('clock', 0.0)]

"proctor opened their"
   count (used 2-word context): [('books', 0.5), ('laptops', 0.25), ('exams', 0.25), ('proctor', 0.0)]
   neural: [('clock', 0.524), ('exam', 0.459), ('exams', 0.004), ('laptops', 0.003)]

"teacher closed their"
   count (used 2-word context): [('books', 1.0), ('proctor', 0.0), (